## Three Models to Compare

implementing and evaluating three different classification approaches, each with unique strenghts:

- Logistic Regression - The baseline model using linear decision boundaries
- K-Nearest Neighbors- Distance-based predictions that capture local patterns
- Decision Tree- Rule-based learning for interpretable decisions

## Comprehensize Evaluation

going beyond simple accuracy to understand model performance deeply:

- Confusion matrices to visualize prediction patterns
- Precision, Recall, and F1-Score for balanced assessment
- ROC-AUC curves to evaluate ranking ability
- Threshold tuning to match real-world business needs

# Problem Definition

## What Am I Predicting?

### Binary Outcome
Predict one of two classes: 0 or 1
Every prediction falls into exacly one category, making this a fundamental classification task

### Real-World Examples
- Fraud Detection: Fraudulent vs Legitimate
- Customer Churn: Will Leave vs Will Stay
- Email Fitering: Spam vs Not Spam
- Medical Diagnosis: Disease vs Healthy

### Dataset Requirements
For this project, your data should have:
- Tabular format with rows and columns
- A target column containing 0 or 1 labels
- Feature columns (numeric or encoded)

In [3]:
# setup: import libraries and load data

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt

In [ ]:
# load your dataset
df = pd.read_csv("data.csv")
print(df.shape)
df.head()

### Key Setup Steps
1. Confirm your target columns exist and contains binary values
2. Check dataset dimensions to understand scale
3. Preview first few rows to validate data loading
4. Verify no obvious formatting issues

In [ ]:
# quick eda for classification tasks
# essential data quality checks

# check class distribution
df["target"].value_counts(normalize=True)

# find missing values
df.isna().sum().sort_values(
    ascending=False
).head(10)

### What to Look For
- Class Imbalance: Are classes roughly equal or heavily skewed?
- Missing Data: Which feature have gaps?
- Feature Types: Numeric vs Categorical columns

In [ ]:
# define features (X) and target (y)

X = df.drop(columns=["target"])
y = df["target"].astype(int)

print(X.shape, y.shape)

### Critical Best Practices
- Remove Leakage Columns: Drop any feature that contains information from the future or directly reveals target (like transaction IDs, timestamps after the event, or derived target variables)
- Ensure Integer Target: Convert target to int type to avoid errors witgh some sklearn functions
- Verify Shapes: X should have one fewer columns than the original dataframe, and y should have the same number of rows

In [ ]:
# train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# baseline model: Logistic Regression Pipeline

# build a proper pipeline

log_reg = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000
    ))
])

log_reg.fit(X_train, y_train)

### Why Use Pipelines?
- Prevents Data Leakage: Scaler is fit only on training data, then applied to test data
- Clearer Code: Transform and predict in single steps
- Production Ready: Easy to save and deploy as one object
- Reproducibility: Guarantees same preprocessing every time

In [ ]:
# confusion matrix & metrics

y_pred = log_predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n", cm)
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
# probability evaluation with ROC-AUC

# generate probability scores

y_prob = log_reg.predict_proba(X_test[:,1])

auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC:", round(auc, 4))

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.plot(fpr,tpr)
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Logistic)")
plt.show()

### Understanding ROC-AUC
This ROC Curve plots True Postive Rate against False Positive Rate across all possible thresholds.
- AUC = 1.0; Perfect Classifier
- AUC = 0.5; Random guessing (diagonal line)
- AUC > 0.8; generally strong performance

AUC measures how well the model ranks predictions-can it reliably score true positves higher than true negatives?

In [ ]:
# threshold tuning 
def eval_at_threshold(y_true, y_prob, thr):
    y_hat = (y_prob >= thr).astype(int)
    return {
        "threshold": thr,
        "precision": precision_score(y_true, y_hat, zero_division=0),
        "recall": recall_score(y_true, y_hat, zero_division=0),
        "f1": f1_score(y_true, y_hat, zero_division=0)
    }

for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    print(eval_at_threshold(y_true, y_prob, thr))

In [ ]:
# implement KNN classifier

knn = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(
        n_neighbors=7
    ))
])

knn.fit(X_train, y_train)
knn_pred = knn.predict(X_test)

print(classification_report(y_test, knn_pred, digits=3))

### How KNN Works
KNN makes prediction by finding the K closest training examples to each test point, then voting among their labels
- Distance-Based: Users Euclidean distance in feature space
- Non-Parametric: No training phase, just memorizes data
- Local Patterns: Captures complex decision boundaries

In [ ]:
# decision tree for interpretability
tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=10,
    random_state=42
)

tree.fit(X_train, y_train)
tree_pred = test.predict(X_test)

print(classification_report(y_test, tree_pred, digits=3))

### Key Advantages
- Human-interpretable rules
- No scaling required
- Handles mixed data types
- Captures feature interactions

### Watch Out For
- Prone to overfitting without constraints
- Sensitive to small data changes
- Can create overly complex boundaries
- Biased toward features with many levels